In [1]:
%%capture
!pip install unsloth evaluate

In [ ]:
from huggingface_hub import login

hf_token = "hf_***"
login(hf_token)

In [3]:
%%capture
from datasets import load_dataset, DatasetDict
train_ds = load_dataset("ikram98ai/trademark_detection",split='train')
test_ds = load_dataset("ikram98ai/trademark_detection", split='test')
val_ds = load_dataset("ikram98ai/trademark_detection", split='val')

In [4]:
dataset = DatasetDict({
    'train': train_ds,
    'test': test_ds,
    'validation': val_ds
})
dataset.push_to_hub("johnhmeyer123/trademark_detection_dataset")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/17 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/johnhmeyer123/trademark_detection_dataset/commit/12604af4901f00ca47b6cd850f58012d00e5a4c8', commit_message='Upload dataset', commit_description='', oid='12604af4901f00ca47b6cd850f58012d00e5a4c8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/johnhmeyer123/trademark_detection_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='johnhmeyer123/trademark_detection_dataset'), pr_revision=None, pr_num=None)

In [5]:
system_prompt = """You are an expert in trademark identification for apparel designs. Your task is to analyze images of apparel and determine 
    if they contain licensed trademarks such as Greek organization letters (fraternities/sororities) or collegiate/university marks. Your response 
    must strictly follow this two-line format: first indicating 'Licensed trademarks detected: Yes' or 'Licensed trademarks detected: No', followed 
    by 'Organization:' with either the specific organization/university name(s) identified or 'None' if no trademarks are detected."""

instruction = """Examine these apparel images and identify if they contain licensed marks or Greek letters. If yes, name the Greek organization or university associated."""
    

In [6]:
import requests
from PIL import Image as PILImage
from io import BytesIO

def load_image_from_url(url):
    """Helper function to download and convert image from URL"""
    try:
        response = requests.get(url, stream=True, timeout=10)
        response.raise_for_status()
        return PILImage.open(BytesIO(response.content)).convert("RGB")
    except Exception as e:
        print(f"Error loading image from {url}: {str(e)}")
        return None


In [7]:
%%capture
from unsloth import FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "ikram98ai/trademark_detection_qwen7b_unsloth_lora", 
    load_in_4bit = True, 
)

2025-05-09 20:07:52.062638: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746821272.337131      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746821272.411586      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [8]:
for sample in test_ds.batch(3):
    print(sample)
    break

Batching examples:   0%|          | 0/8261 [00:00<?, ? examples/s]

{'image_urls': [['https://res.cloudinary.com/dsnai3oon/image/upload/v1740659135/cropped_images/ff406624c7db4aa52c98c880084b50ec_1740659134.jpg'], ['https://cf.freshprints.com/designs/1707337624424ddmsu_nt_front.png', 'https://cf.freshprints.com/designs/1707337624424aviks_nt_back.png'], ['https://cf.freshprints.com/designs/1725038509949rfkyj_nt_front.png', 'https://cf.freshprints.com/designs/1725038509949awhps_nt_back.png']], 'trademark_detected': ['Yes', 'Yes', 'Yes'], 'organization': ['Northeastern University', 'Zeta Tau Alpha', 'Alpha Chi Omega']}


In [9]:
import evaluate
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

In [18]:
FastVisionModel.for_inference(model) # Enable for inference!


predictions = []
references = []
for i, batch in enumerate(test_ds.batch(10)):
    if i >10:
        break
    print('batch no#: ',i)
    
    preds = []
    for sample in batch['image_urls']:
        message = [
            {"role": "user", "content": [{"type": "image"} for _ in sample]
             +[
                {"type": "text", "text": system_prompt + '\n\n' + instruction}
            ]}
        ]

        input_text = tokenizer.apply_chat_template(message, add_generation_prompt = True)
        inputs = tokenizer(
            [load_image_from_url(img_url) for img_url in sample],
            input_text,
            add_special_tokens = False,
            return_tensors = "pt",
        ).to("cuda")

        generated_text = model.generate(**inputs,max_new_tokens = 128, use_cache = True, temperature = 1.5, min_p = 0.1)
        pred = tokenizer.decode(generated_text[0]).split("Licensed trademarks detected: ")[-1].split("\n")[0]
        preds.append(pred)
        
    predictions.extend(preds)
    references.extend(batch['trademark_detected'])
  
    

batch no#:  0
batch no#:  1
batch no#:  2
batch no#:  3
batch no#:  4
batch no#:  5
batch no#:  6
batch no#:  7
batch no#:  8
batch no#:  9
batch no#:  10


In [19]:
actuals = [int(str(ref).lower().strip()=="yes") for ref in references]
preds = [int(str(pred).lower().strip()=="yes") for pred in predictions]

In [20]:
len(list(zip(references,predictions))), list(zip(references,predictions))

(110,
 [('Yes', 'No'),
  ('Yes', 'No'),
  ('Yes', 'Yes'),
  ('No', 'No'),
  ('Yes', 'Yes'),
  ('No', 'No'),
  ('No', 'No'),
  ('No', 'No'),
  ('Yes', 'Yes'),
  ('Yes', 'No'),
  ('No', 'No'),
  ('No', 'No'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('No', 'No'),
  ('No', 'No'),
  ('Yes', 'Yes'),
  ('No', 'No'),
  ('No', 'No'),
  ('Yes', 'Yes'),
  ('No', 'No'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('No', 'No'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('No', 'No'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('No', 'Yes'),
  ('No', 'No'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('Yes', 'No'),
  ('No', 'No'),
  ('No', 'No'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('No', 'No'),
  ('Yes', 'Yes'),
  ('No', 'No'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('No', 'No'),
  ('No', 'No'),
  ('Yes', 'Yes'),
  ('Yes', 'Yes'),
  ('No', 'No'),
  ('No', 'Yes'),
  ('No', 'No'),
  ('No', 'No'),
  

In [21]:
clf_metrics.compute(references=actuals, predictions=preds)

{'accuracy': 0.8909090909090909,
 'f1': 0.9,
 'precision': 0.8852459016393442,
 'recall': 0.9152542372881356}

In [22]:

model.push_to_hub("johnhmeyer123/trademark_detection_lora_model",token= hf_token)
tokenizer.push_to_hub("johnhmeyer123/trademark_detection_lora_model", token= hf_token)

README.md:   0%|          | 0.00/614 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/206M [00:00<?, ?B/s]

Saved model to https://huggingface.co/johnhmeyer123/trademark_detection_lora_model


tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]